In essence, we cannot “solve” instances of LWE. But estimating how hard they are is slightly more studied. I think going for a solver is way-way beyond my scope, thus https://github.com/malb/lattice-estimator by Albrecht et al will come in handy.

In [ ]:
from sage.crypto.lwe import samples

In [ ]:
S = samples(4, 4, 'Regev')
S

In [ ]:
# importing submodules with hyphens: https://stackoverflow.com/questions/8350853/
import sys
sys.path.append('lattice-estimator')
import estimator

In [ ]:
estimator.LWE.Estimate.rough

## Lattice estimator usage with built-in parameter sets

In [ ]:
%%time

estimator.schemes.Kyber512

In [ ]:
%%time
estimator.LWE.primal_usvp(estimator.schemes.Kyber512)

In [ ]:
%%time
estimator.LWE.estimate.rough(estimator.schemes.Kyber512)

In [ ]:
%%time
estimator.LWE.estimate(estimator.schemes.Kyber512)

## Lattice estimator with our own parameters
Let's use Regev's methodology for choosing parameters; let's use base parameters similar to Kyber512

In [ ]:
S = samples(m=512, n=512, lwe='Regev', seed=4815)

In [ ]:
next_prime(pow(512, 2))

In [ ]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler
# this comes very directly from sage.crypto.lwe.Regev


###### NOTE -- I don't think I have seen any paper use this “Both Gaussian” form. Trying just for funsies!
#choose n - nr of variables
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
# Chi-s - secret distribution
Xs = estimator.ND.DiscreteGaussian(sigma, arith_mean)
# Chi-e - error distribution
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with Gaussian Xs and Gaussian Xe'

#optionally define how many samples we want to limit the oracle to
#sage.all.oo = infinity
m = oo

In [ ]:
#attempt 0:
# estimator.lwe_parameters.LWEParameters(n, q, Xs, Xe, m, tag)
#got back:
# AttributeError: '...Sampler' object has no attribute 'resize'
#meaning we probably need to wrap sampler in a latticeEstimator compatible object (from package estimator.nd)

#attempt 1:
Regev_GaussXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_GaussXs_GaussXe

In [ ]:
# the most standard form of Regev: uniform secret, Gaussian error
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
Xs = estimator.ND.UniformMod(q)
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with uniform Xs and Gaussian Xe'
m = oo

Regev_UniXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_GaussXe

In [ ]:
n = 512
q = ZZ(next_prime(n**2))
# Chi-s - secret distribution
Xs = estimator.ND.UniformMod(q)
# Chi-e - error distribution
##### manually chose modulus 496 as this would give parameter \sigma close to those with Gaussian distribution
Xe = estimator.ND.UniformMod(ZZ(495))
tag = 'Regev with uniform Xs and uniform Xe'
m = oo

Regev_UniXs_UniXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_UniXe

### Estimations on our own samples!

In [ ]:
deny_list=["arora-gb", "bkw", "bdd", "bdd_hybrid", "bdd_mitm_hybrid", "dual"]

In [ ]:
%%time
estimator.LWE.estimate(Regev_GaussXs_GaussXe, deny_list=deny_list)

In [ ]:
%%time
estimator.LWE.estimate(Regev_UniXs_GaussXe)#, deny_list=deny_list)

In [ ]:
%%time
estimator.LWE.estimate(Regev_UniXs_UniXe, deny_list=deny_list)

It looks like sometimes the estimator can run for a lot-lot longer than expected. `UniXs_GaussXe` finished in 9.74 seconds wall time, while `UniXs_UniXe` did not estimate in a few minutes... Restarted kernel, then waited 24 minutes, did not finish. Swapped to `estimate.rough()` (from just `estimate()`).

---

Looking back, it was clear what the error was -- the choice of uniform distribution means that going for uniform errors in the entirety of $q$ would be ridiculously difficult. Thus, errors should probably be centered in some smaller subsection. I went back and modified the selection such that my errors would be from an odd modulus which would result a similar $\sigma$ value for the error distribution (in our case, original Gaussian $\chi_e$ had $\sigma\approx 143.03$; uniform $\chi_e$ with modulus $495$ gave $\sigma\approx 142.89$).

## Side-question

It seems that for the noise distribution, stddev is roughly one of the few parameters that matters. Coincidence? Maybe is it the case that for dimensions they care about, these distributions do not differ much?

Maybe the approach for this is to find all the different LWE schemes that lattice-estimator supports? -- Nope, that would be way too many schemes. Rather, maybe try to create a table of what each thing supports.

Information gathered from the lattice-estimator `schemes.py` file -- just so that collecting would be easier. Only focusing on schemes which seemingly have a real implementation behind them.

For Regev and LP-style parameters, used `sage.crypto.lwe.<scheme>` with input values for $n$.

| Scheme           | n     | q       | m (# max samples) | Xs                    | Xe                       |
|------------------|-------|---------|-------------------|-----------------------|--------------------------|
| Kyber512         | 2*256 | 3329    | 2*256             | CenteredBinom(3)      | CenteredBinom(3)         |
| Kyber768         | 3*256 | 3329    | 3*256             | CenteredBinom(2)      | CenteredBinom(2)         |
| Kyber1024        | 4*256 | 3329    | 4*256             | CenteredBinom(3)      | CenteredBinom(3)         |
| LightSaber       | 2*256 | 8192    | 2*256             | CenteredBinom(5)      | UniformMod(8)            |
| Saber            | 3*256 | 8192    | 3*256             | CenteredBinom(4)      | UniformMod(8)            |
| FireSaber        | 4*256 | 8192    | 4*256             | CenteredBinom(3)      | UniformMod(8)            |
| Frodo640         | 640   | 32768   | 640 + 16          | DiscreteGaussian(2.8) | DiscreteGaussian(2.8)    |
| Frodo976         | 976   | 65536   | 976 + 16          | DiscreteGaussian(2.3) | DiscreteGaussian(2.3)    |
| Frodo1344        | 1344  | 65536   | 1344 + 16         | DiscreteGaussian(3.0) | DiscreteGaussian(3.0)    |
| Regev-esque-512  | 2*256 | 262147  | -                 | Uniform               | DiscreteGaussian(57.06)  |
| Regev-esque-768  | 3*256 | 589829  | -                 | Uniform               | DiscreteGaussian(92.42)  |
| Regev-esque-1024 | 4*256 | 1048583 | -                 | Uniform               | DiscreteGaussian(130.72) |
| LP-esque-512     | 2*256 | 4099    | 1152              | DiscreteGaussian(…)   | DiscreteGaussian(2.9)    |
| LP-esque-768     | 3*256 | 8209    | 1664              | DiscreteGaussian(…)   | DiscreteGaussian(3.76)   |
| LP-esque-1024    | 4*256 | 8209    | 2176              | DiscreteGaussian(…)   | DiscreteGaussian(3.53)   |

Trivially, the Regev-style parameters seem way out of scope. 

I will wholeheartedly admit that at this point, I've arrived at a bit of a wall again. I consulted the Copilot sands, which did mention that it might be worth looking at variance $\alpha = \frac\sigma q$ in the table as well. Additionally:
> Kyber and Saber families are operating with astonishingly small noise compared to classical Regev constructions. This is possible because they exploit module structure and have much tighter engineering constraints.

Sands also propose testing a hypothesis:
> Holding (n,q,\alpha) approximately fixed, replacing Gaussian, centered binomial, and bounded uniform error distributions produces only minor changes in estimated attack cost.

And finally:
> ... modern hardness estimates are largely driven by variance (or $\alpha$), whereas distribution shape tends to appear only as a secondary effect except for small-support attacks and proof-related considerations.